# White-beam dose for BM32 micro-Laue

The incident spectrum at the sample comes from **pyXRT** (`xrf_beamline`, https://github.com/STardif/pyXRT): BM32 bending magnet, Ir mirrors, Be windows, slits traced back to the source, and the 3.3 m incident air path. It returns an absolute spectral flux (photons/s/eV), which `dose_estimation` integrates against μ_en/ρ(E) with beam hardening through the sample.

**Requirements:** a clone of pyXRT and `pip install xraylib`. Set `PYXRT_PATH` below (or the environment variable of that name).

**Caveats.** pyXRT assumes ideal mirror reflectivities and takes the slit gaps literally, so check the total flux against a diode or ion-chamber reading when you have one, and rescale with `spectrum.scaled_to(measured_flux)`. The KB spot size below is a placeholder: use the measured one.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from dose_estimation import Beam, Spectrum, Material, Layer, Stack, raster_factor
from dose_estimation.sources import bm32_white_beam, bm32_white_beam_from_h5
from dose_estimation.plotting import plot_spectrum

PYXRT_PATH = "~/pyXRT"            # <- your clone of https://github.com/STardif/pyXRT

## 1. Spectrum at the sample

Defaults are the 2026 micro-Laue slit settings. Pass any `BM32_WHITE_BEAM` keyword (`vg3=`, `hg3=`, `current_a=`, `air_path_m=`, ...), or read them from a scan with `bm32_white_beam_from_h5("scan.h5", scan="1.1", pyxrt_path=PYXRT_PATH)`.

In [ ]:
spectrum = bm32_white_beam(pyxrt_path=PYXRT_PATH)          # 3-40 keV, 25 eV steps
# spectrum = spectrum.scaled_to(5e11)                       # if you measured the total flux
lo, hi = spectrum.band(0.01)
print(f"total flux {spectrum.total_flux:.3e} ph/s, power {spectrum.power_w*1e3:.2f} mW, "
      f"mean energy {spectrum.mean_energy_ev/1e3:.1f} keV, band (1 % of peak) {lo/1e3:.1f}-{hi/1e3:.1f} keV")
ax = plot_spectrum(spectrum)
ax.set_title("BM32 white beam at the sample (pyXRT)")

## 2. A bulk crystal in micro-Laue geometry

Reflection geometry, beam at 40° from the surface (`grazing_angle_deg=40`), semi-infinite sample (`thickness_um=np.inf`). For a semi-infinite layer, `dose_rate` is averaged over the first attenuation length and `dose_rate_max` is the entrance-face value at the beam centre.

With a sub-micron focus the photoelectrons (ranges of 1–10 µm here) carry much of the energy out of the footprint, so the kerma overestimates the dose where the beam is. `electron_escape=True` applies the estimate described in `dose_estimation/escape.py` (detour factor fitted to an electron Monte Carlo); `detour_factor=0.4` reproduces the smaller benefit seen in micro-beam *damage* experiments. Read the columns as a bracket, not as one number.

In [ ]:
kb_beam = Beam(spectrum, width_um=0.5, height_um=0.5, profile="gaussian")   # placeholder spot size (FWHM)

samples = {
    "Si": Material("Si", 2.33, name="Si"),
    "Ge": Material("Ge", 5.32, name="Ge"),
    "NMC811": Material("LiNi0.8Mn0.1Co0.1O2", 4.8, name="NMC811"),
    "UO2": Material("UO2", 10.97, name="UO2"),
}
rows = []
for key, mat in samples.items():
    st = Stack([Layer.solid(key, mat, np.inf)], grazing_angle_deg=40)
    k = st.compute(kb_beam).components[0]
    e4 = st.compute(kb_beam, electron_escape=True, detour_factor=0.4).components[0]
    e10 = st.compute(kb_beam, electron_escape=True).components[0]
    rows.append((key, k.dose_rate_max / 1e6, e4.dose_rate_max / 1e6, e10.dose_rate_max / 1e6))
print(f"{'sample':8s} {'kerma':>12s} {'detour 0.4':>12s} {'default':>12s}   peak dose rate, MGy/s")
for r in rows:
    print(f"{r[0]:8s} {r[1]:12.4g} {r[2]:12.4g} {r[3]:12.4g}")

## 3. Where in the spectrum does the dose come from?

`spectral_contribution` splits the dose rate over the spectral lines. Edges of the sample (Ni K at 8.3 keV, Ge K at 11.1 keV) and the mirror cut-off shape it; low-energy photons count for more than their share of the flux.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
de_ = np.gradient(spectrum.energy_ev)
for key in ["Si", "Ge", "NMC811"]:
    st = Stack([Layer.solid(key, samples[key], np.inf)], grazing_angle_deg=40)
    r = st.compute(kb_beam)
    e, c = r.spectral_contribution(key, key, peak=True)
    ax.plot(e / 1e3, c / de_ / c.sum() * 1e3, label=key)
ax.plot(spectrum.energy_ev / 1e3, spectrum.density() / spectrum.total_flux * 1e3, "k:", label="photon flux")
ax.set_xlabel("energy (keV)"); ax.set_ylabel("fraction per keV"); ax.legend(fontsize="small")

## 4. A battery electrode under the white beam

A porous NMC811 electrode (30 µm, 30 % electrolyte) on 20 µm Al, in the same 40° geometry. The table gives the beam-on time to reach the dose thresholds of the 2023 paper (no damage below 80 kGy; apparent microstructure changes from ~1 MGy; kinetic limitation from ~11 MGy), at the beam centre.

In [ ]:
ely = Material.solution(Material("C3H6O3", 1.07, name="DMC"), "LiPF6", 1.0, density=1.25, name="electrolyte")
electrode = Stack([Layer.porous("cathode", samples["NMC811"], porosity=0.30, filler=ely, thickness_um=30),
                   Layer.solid("Al", Material("Al", 2.7, name="Al"), 20)], grazing_angle_deg=40)
r = electrode.compute(kb_beam, electron_escape=True)
print(r.summary("MGy"))
print()
for thr in (80e3, 1e6, 11e6):
    t_nmc = r.time_to_dose(thr, "cathode", "NMC811")
    t_ely = r.time_to_dose(thr, "cathode", "electrolyte")
    t_mix = r.time_to_dose(thr, "cathode")
    print(f"{thr/1e6:6.2f} MGy: NMC {t_nmc*1e3:9.3g} ms | electrolyte own {t_ely*1e3:9.3g} ms | "
          f"electrons shared {t_mix*1e3:9.3g} ms")

## 5. Maps

For a raster map with a dwell time `t` per point, the dose averaged over the mapped area once the map is done is `dose_rate * t * raster_factor(beam, dx, dy)`. Steps smaller than the beam make neighbouring exposures overlap.

In [ ]:
c = r.get("cathode", "NMC811")
for step in (0.25, 0.5, 1.0, 2.0):
    f = raster_factor(kb_beam, step, step)
    print(f"step {step:4.2f} um: map-averaged NMC dose for 1 s/point = {c.dose_rate * 1.0 * f / 1e6:10.4g} MGy")

## 6. Escape estimate versus electron Monte Carlo

`electron_escape="mc"` replaces the straight-track estimate by an electron-transport Monte Carlo (screened Rutherford scattering, Joy–Luo stopping power, Sauter emission, free surface) on a few energies of the spectrum. The default analytic estimate uses a detour factor fitted to this Monte Carlo, so the two should agree within ~10 %; `validation/escape_crosscheck.py` has the full comparison.

In [ ]:
for key in ["Si", "NMC811"]:
    st = Stack([Layer.solid(key, samples[key], np.inf)], grazing_angle_deg=40)
    an = st.compute(kb_beam, electron_escape=True).components[0]
    mc = st.compute(kb_beam, electron_escape="mc").components[0]
    print(f"{key:7s} retention (signal-weighted): analytic {an.retention_mean:.3f}, Monte Carlo {mc.retention_mean:.3f}; "
          f"peak: {an.retention_peak:.3f} vs {mc.retention_peak:.3f}")

## 7. Temperature rise

A few mW absorbed in a sub-micron spot: the steady temperature rise is set by the conductivity near the spot. Bulk crystals conduct well; a porous electrode filled with electrolyte does not (k ≈ 1 W/m/K assumed below).

In [ ]:
from dose_estimation import thermal
for key, k in [("Si", 148), ("Ge", 60), ("NMC811", 10), ("UO2", 8.7)]:
    st = Stack([Layer.solid(key, samples[key], np.inf)], grazing_angle_deg=40)
    rr = st.compute(kb_beam)
    print(f"{key:7s} k = {k:5g} W/m/K: {thermal.temperature_rise(rr, k).total_k:6.2f} K")
est = thermal.temperature_rise(r, {"cathode": 1.0, "Al": 237}, sink_radius_um=2000)
print("electrode:", est.summary())

## 8. Uncertainty

The absolute pyXRT flux and the KB spot size are the weakest inputs here; 30 % and 20 % (1σ) are placeholders until a diode reading and a knife-edge scan are at hand.

In [ ]:
from dose_estimation import propagate_stack
u = propagate_stack(electrode, kb_beam, exposure_s=1.0, flux_rel=0.30, size_rel=0.20, n=100,
                    electron_escape=True)
print(u.summary(1e-6, "MGy  per second of exposure"))